FLOPS Analysis

In [ ]:
# 本 notebook 依赖 thop（用于统计模型的 MACs/FLOPs）和 torch
# 这里先打印出两者的版本号，方便排查环境问题
from importlib.metadata import version

pkgs = [
    "thop",
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

Simple benchmark with fixed batch size

In [ ]:
import torch
from thop import profile

# For installation instructions, see:
# https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# 【bug 修正】原代码为 `from Build_an_LLM_from_Scratch.ch04 import *`，
# 这是一个不存在的包名（笔误），实际应导入本仓库真正的包 llms_from_scratch，已修正为下面这行
from llms_from_scratch.ch04 import *


# BASE_CONFIG：GPT 模型的公共超参数配置
# 注意这里把 drop_rate 设为 0.0，是因为 FLOPs 分析只关心前向/反向传播中的乘加运算量，
# Dropout 本身不涉及可训练参数、也不产生显著的 FLOPs，关闭它可以让基准测试更简洁
BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

# 四种不同规模的 GPT-2 模型配置（嵌入维度、层数、注意力头数）
# 模型越大，参数量和 FLOPs 通常呈近似线性/超线性增长，用来对比不同规模模型的计算开销
model_configs = {
    "gpt-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# 优先使用 GPU（CUDA），没有 GPU 则退回 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 2
# 构造一个随机 token id 张量作为输入，用于让 thop.profile 追踪一次前向传播的计算量
# 形状为 (batch_size, context_length)，取值范围覆盖整个词表 [0, vocab_size)
input_tensor = torch.randint(0, 50257, (batch_size, 1024)).to(device)

for size in model_configs:
    # 用当前规模的超参数覆盖 BASE_CONFIG 中的对应字段
    BASE_CONFIG.update(model_configs[size])

    # 【bug 修正】原代码为 `model = *(BASE_CONFIG).bfloat16()`，
    # 这里 `*(...)` 是非法的星号解包写法，直接运行会触发 SyntaxError，
    # 根据上下文（构造 GPT 模型并转为 bfloat16）应改为用 GPTModel 类实例化配置字典
    # 使用 bfloat16 半精度可以减小显存占用、加快计算，同时基本不影响 FLOPs 的计数方式
    model = GPTModel(BASE_CONFIG).bfloat16()
    model.to(device)

    # MACS = multiply-accumulate operations
    # MACS are typically counted as two FLOPS (one multiply and one accumulate)
    # 即：1 次乘加(MAC) = 1 次乘法 + 1 次加法 = 2 次浮点运算(FLOPs)
    # thop.profile 会对模型做一次前向传播（inputs=(input_tensor,)），统计出总的 MACs 和参数量 params
    macs, params = profile(model, inputs=(input_tensor,), verbose=False)
    # 将 MACs 换算为 FLOPs：乘以 2
    flops = 2*macs
    print(f"{size:18}: {flops:.1e} FLOPS")

    # 释放当前模型占用的显存，避免多次循环后显存累积溢出
    del model
    torch.cuda.empty_cache()

Simple benchmark with automatic batch size finding

In [ ]:
for size in model_configs:
    print(f"\nProcessing {size}")
    config = BASE_CONFIG.copy()
    config.update(model_configs[size])

    # 用二分查找（binary search）的思路，在显存允许的范围内寻找最大可用 batch size：
    # min_batch_size / max_possible_batch_size 分别是搜索区间的下界和当前上界，
    # max_batch_size 记录目前为止成功跑通的最大 batch size
    min_batch_size = 1
    max_batch_size = None
    max_possible_batch_size = 4096

    while min_batch_size <= max_possible_batch_size:
        # 每轮取区间中点作为待测试的 batch size
        batch_size = (min_batch_size + max_possible_batch_size) // 2
        try:
            input_tensor = torch.randint(
                0, config["vocab_size"],
                (batch_size, config["context_length"]),
                device=device
            )

            model = GPTModel(config).bfloat16().to(device)

            # MACS = multiply-accumulate operations
            # MACS are typically counted as two FLOPS (one multiply and one accumulate)
            # 同样地，1 MAC = 2 FLOPs，这里统计的是在当前 batch_size 下完整前向传播的 FLOPs
            macs, params = profile(model, inputs=(input_tensor,), verbose=False)
            flops = 2 * macs
            print(f"  Batch size {batch_size}: {flops:.1e} FLOPS")

            # 当前 batch size 能成功跑通（未 OOM），说明还有余量，
            # 把搜索下界上移，尝试更大的 batch size
            min_batch_size = batch_size + 1
            max_batch_size = batch_size

            # Clean up
            # 及时释放显存，避免影响下一轮的显存判断
            del model, input_tensor
            torch.cuda.empty_cache()

        except RuntimeError as e:
            if "out of memory" in str(e):
                # 显存不足（OOM），说明当前 batch size 太大，
                # 把搜索上界下移，缩小 batch size 继续尝试
                max_possible_batch_size = batch_size - 1

                # Clean up
                try:
                    del model, input_tensor
                    torch.cuda.empty_cache()
                except NameError:
                    pass
            else:
                # 不是显存不足导致的错误，直接抛出，避免掩盖其他真正的问题
                raise e

Benchmark with automatic batch size finding and Model FLOP Utilization (MFU)

In [ ]:
# Theoretical max flops per second provided by the GPU manufacturer
# 下面这份字典记录了几款常见 GPU 在不同精度（FP32 / FP16 / BF16）下，
# 厂商标称的理论峰值算力（单位：FLOPs/秒），数据来源见各行注释链接（techpowerup.com）
# 后面计算 MFU（Model FLOPs Utilization，模型算力利用率）时，会用它作为“理论最大值”的分母

flops_per_second = {
    # https://www.techpowerup.com/gpu-specs/h100-pcie-80-gb.c3899
    "H100": {
        torch.float32: 51.22e12,  # 51.22 TFLOPs for FP32 on NVIDIA H100
        torch.float16: 204.9e12,  # 204.9 TFLOPs for FP16 on NVIDIA H100
        torch.bfloat16: 204.9e12
    },
    # https://www.techpowerup.com/gpu-specs/l4.c4091
    "L4": {
        torch.float32: 30.29e12,  # 30.29 TFLOPs for FP32 on NVIDIA L4
        torch.float16: 30.29e12,  # 30.29 TFLOPs for FP16 on NVIDIA L4
        torch.bfloat16: 30.29e12
    },
    # https://www.techpowerup.com/gpu-specs/tesla-t4.c3316
    "T4": {
        torch.float32: 8.1e12,  # 8.1 TFLOPs for FP32 on NVIDIA T4
        torch.float16: 65.13e12,  # 65.13 TFLOPs for FP16 on NVIDIA T4
        torch.bfloat16: 65.13e12
    },
    # https://www.techpowerup.com/gpu-specs/a10g.c3798
    "A10G": {
        torch.float32: 31.52e12,  # 31.52 TFLOPs for FP32 on NVIDIA A10G
        torch.float16: 31.52e12,  # 31.52 TFLOPs for FP16 on NVIDIA A10G
        torch.bfloat16: 31.52e12
    },
    # https://www.techpowerup.com/gpu-specs/a100-pcie-40-gb.c3623
    "A100": {
        torch.float32: 19.49e12,  # 19.49 TFLOPs for FP32 on NVIDIA A100
        torch.float16: 77.97e12,  # 77.97 TFLOPs for FP16 on NVIDIA A100
        torch.bfloat16: 77.97e12
    },
    # https://www.techpowerup.com/gpu-specs/geforce-rtx-3080.c3621
    "RTX_3080": {
        torch.float32: 29.77e12,  # 29.77 TFLOPs for FP32 on NVIDIA RTX 3080
        torch.float16: 29.77e12,  # 29.77 TFLOPs for FP16 on NVIDIA RTX 3080
        torch.bfloat16: 29.77e12
    },
    # https://www.techpowerup.com/gpu-specs/geforce-rtx-3090.c3622
    "RTX_3090": {
        torch.float32: 35.58e12,  # 35.58 TFLOPs for FP32 on NVIDIA RTX 3090
        torch.float16: 35.58e12,  # 35.58 TFLOPs for FP16 on NVIDIA RTX 3090
        torch.bfloat16: 35.58e12
    }
}

In [ ]:
import time

def get_gpu_model(flops_per_second_dict):
    # 通过 torch.cuda.get_device_name 得到当前 GPU 的完整设备名字符串，
    # 再和 flops_per_second 字典里的 key（GPU 型号简称）做子串匹配，找出对应型号
    device_name = torch.cuda.get_device_name(0)
    for model in flops_per_second_dict.keys():
        if model in device_name:
            return model
    return "Unknown"  # Default if no matching model is found


gpu_model = get_gpu_model(flops_per_second)
print("GPU Model:", gpu_model)

if gpu_model != "Unknown":

    for size in model_configs:
        print(f"\nProcessing {size}")
        config = BASE_CONFIG.copy()
        config.update(model_configs[size])

        # 同样通过二分查找，先找出当前模型规模下显存允许的最大 batch size
        min_batch_size = 1
        max_batch_size = None
        max_possible_batch_size = 4096

        while min_batch_size <= max_possible_batch_size:
            batch_size = (min_batch_size + max_possible_batch_size) // 2
            try:
                input_tensor = torch.randint(
                    0, config["vocab_size"],
                    (batch_size, config["context_length"]),
                    device=device
                )

                model = GPTModel(config).bfloat16().to(device)
                # 切换到训练模式，因为下面要做反向传播（backward），需要构建计算图
                model.train()

                # Start timing
                # torch.cuda.synchronize() 用于等待 GPU 上所有已提交的核函数执行完毕，
                # 这样才能准确测量“墙钟时间”（wall-clock time），避免异步执行导致计时不准
                torch.cuda.synchronize()
                start_time = time.time()

                # Forward & backward pass
                # 真正执行一次前向传播 + 反向传播，用来测量实际耗时
                output = model(input_tensor)
                loss = output.sum()  # Compute a dummy loss
                loss.backward()

                # End timing
                torch.cuda.synchronize()
                end_time = time.time()

                total_time_seconds = end_time - start_time

                # Calculate FLOPs for forward pass
                # 用 thop.profile 单独统计一次前向传播的 MACs，再换算成 FLOPs（乘以 2）
                macs, params = profile(model, inputs=(input_tensor,), verbose=False)
                flops_forward = 2 * macs  # Assuming one MAC equals two FLOPs

                # Estimate FLOPs for backward pass (typically 2x forward FLOPs)
                # 反向传播没有直接的工具统计，这里采用业界常用的经验估算：
                # 反向传播的计算量约为前向传播的 2 倍（因为要对每个参数分别计算相对于输入和权重的梯度）
                flops_backward = 2 * flops_forward

                # Total FLOPs for forward + backward passes
                # 前向 + 反向的总 FLOPs，等价于 3 倍前向 FLOPs（1x 前向 + 2x 反向）
                total_flops = flops_forward + flops_backward  # Or total_flops = flops_forward * 3

                # 获取模型参数的数据类型（如 bfloat16），用于查表得到该精度下 GPU 的理论峰值算力
                data_type = next(model.parameters()).dtype
                max_flops_per_second = flops_per_second[gpu_model].get(data_type, 0)

                # Compute tokens per second
                # 本次前向+反向总共处理的 token 数 = batch_size * 上下文长度
                tokens_processed = batch_size * config["context_length"]
                # 实测的吞吐量：每秒处理多少个 token
                tokens_per_second = tokens_processed / total_time_seconds

                # Compute FLOPs per token
                # 平均每个 token 需要多少 FLOPs（用总 FLOPs 除以总 token 数）
                flops_per_token = total_flops / tokens_processed

                # Compute theoretical max tokens per second
                # 用 GPU 的理论峰值算力除以“每 token 所需 FLOPs”，
                # 得到在算力被 100% 利用的理想情况下，每秒最多能处理多少 token
                if flops_per_token > 0:
                    theoretical_max_tokens_per_second = max_flops_per_second / flops_per_token
                else:
                    theoretical_max_tokens_per_second = 0  # Avoid division by zero

                # Compute MFU
                # MFU（Model FLOPs Utilization，模型算力利用率）
                # = 实际吞吐量 / 理论最大吞吐量
                # 该指标反映了训练/推理过程对 GPU 算力的实际利用效率，越接近 1 说明利用率越高
                if theoretical_max_tokens_per_second > 0:
                    mfu = tokens_per_second / theoretical_max_tokens_per_second
                else:
                    mfu = 0  # Avoid division by zero

                print(f"  Batch size {batch_size}: Tokens/sec: {tokens_per_second:.2f}, MFU: {mfu:.4f}")

                # If successful, try a larger batch size
                min_batch_size = batch_size + 1
                max_batch_size = batch_size

                # Clean up
                del model, input_tensor, output, loss
                torch.cuda.empty_cache()

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    # Try smaller batch size
                    max_possible_batch_size = batch_size - 1

                    # Clean up
                    try:
                        del model, input_tensor
                        torch.cuda.empty_cache()
                    except NameError:
                        pass
                else:
                    raise e

else:
    # 如果当前 GPU 型号不在 flops_per_second 字典中，无法查到理论峰值算力，
    # 就无法计算 MFU，需要用户手动补充该 GPU 的算力信息
    print("Unknown GPU model. Please update the flops_per_second dictionary with your GPU information.")